In [ ]:
import os
import json

# Replace these with your actual Kaggle username and API key
kaggle_credentials = {
    "username": "your kaggle username here ",
    "key": "your api key here"
}

# Create the file
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_credentials, f)

os.chmod('/root/.kaggle/kaggle.json', 600)
print("Done!")

Done!


In [2]:
!pip install kaggle -q
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:21<00:00, 189MB/s]



Unzip the Dataset

In [3]:
!unzip -q 140k-real-and-fake-faces.zip -d dataset

Dataset structure

In [4]:
import os

for root, dirs, files in os.walk('dataset'):
    level = root.replace('dataset', '').count(os.sep)
    if level < 3:
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/ ({len(os.listdir(root))} items)')

dataset/ (4 items)
  real_vs_fake/ (1 items)
    real-vs-fake/ (3 items)


Train/Test Split

In [5]:
for root, dirs, files in os.walk('dataset'):
    level = root.replace('dataset', '').count(os.sep)
    if level < 4:
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/ ({len(os.listdir(root))} items)')

dataset/ (4 items)
  real_vs_fake/ (1 items)
    real-vs-fake/ (3 items)
      train/ (2 items)
      valid/ (2 items)
      test/ (2 items)


In [6]:
import os

base = 'dataset/real_vs_fake/real-vs-fake'

for split in ['train', 'valid', 'test']:
    for label in ['real', 'fake']:
        path = os.path.join(base, split, label)
        count = len(os.listdir(path))
        print(f'{split}/{label}: {count} images')

train/real: 50000 images
train/fake: 50000 images
valid/real: 10000 images
valid/fake: 10000 images
test/real: 10000 images
test/fake: 10000 images


In [7]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# Define image transformations
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Load datasets
base = 'dataset/real_vs_fake/real-vs-fake'

train_dataset = ImageFolder(root=f'{base}/train', transform=train_transform)
val_dataset   = ImageFolder(root=f'{base}/valid', transform=val_transform)
test_dataset  = ImageFolder(root=f'{base}/test',  transform=val_transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2)

# Check class mapping
print("Classes:", train_dataset.classes)
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

Classes: ['fake', 'real']
Train batches: 3125
Val batches: 625


In [8]:
import torch
import torch.nn as nn
from torchvision import models

# Load pretrained EfficientNet-B0
model = models.efficientnet_b0(weights='IMAGENET1K_V1')

# Freeze all base layers (we don't retrain them, just use their features)
for param in model.parameters():
    param.requires_grad = False

# Replace the classifier head with our own
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(1280, 256),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(256, 1),
    nn.Sigmoid()
)

# Move model to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print("Device:", device)
print("Model ready!")

# Quick sanity check — pass one fake batch through
sample_batch = torch.randn(4, 3, 224, 224).to(device)
output = model(sample_batch)
print("Output shape:", output.shape)
print("Sample outputs:", output.detach().cpu().numpy().flatten())

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 55.9MB/s]


Device: cuda
Model ready!
Output shape: torch.Size([4, 1])
Sample outputs: [0.48059583 0.48101434 0.47464064 0.48975936]


In [9]:
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

# Loss function and optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)
scheduler = StepLR(optimizer, step_size=3, gamma=0.5)

# Training function
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        predicted = (outputs >= 0.5).float()
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        # Print progress every 200 batches
        if (batch_idx + 1) % 200 == 0:
            print(f'  Batch {batch_idx+1}/{len(loader)} | Loss: {loss.item():.4f}')

    return total_loss / len(loader), correct / total


# Validation function
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            predicted = (outputs >= 0.5).float()
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total


# Run training for 5 epochs
EPOCHS = 5

for epoch in range(EPOCHS):
    print(f'\nEpoch {epoch+1}/{EPOCHS}')
    print('-' * 40)

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    scheduler.step()

    print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%')
    print(f'Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc*100:.2f}%')


Epoch 1/5
----------------------------------------
  Batch 200/3125 | Loss: 0.4856
  Batch 400/3125 | Loss: 0.5727
  Batch 600/3125 | Loss: 0.5211
  Batch 800/3125 | Loss: 0.5060
  Batch 1000/3125 | Loss: 0.5477
  Batch 1200/3125 | Loss: 0.4899
  Batch 1400/3125 | Loss: 0.3818
  Batch 1600/3125 | Loss: 0.6723
  Batch 1800/3125 | Loss: 0.5480
  Batch 2000/3125 | Loss: 0.3975
  Batch 2200/3125 | Loss: 0.5302
  Batch 2400/3125 | Loss: 0.5228
  Batch 2600/3125 | Loss: 0.5353
  Batch 2800/3125 | Loss: 0.3653
  Batch 3000/3125 | Loss: 0.3463
Train Loss: 0.4836 | Train Acc: 76.67%
Val Loss:   0.3714 | Val Acc:   83.60%

Epoch 2/5
----------------------------------------
  Batch 200/3125 | Loss: 0.4364
  Batch 400/3125 | Loss: 0.4768
  Batch 600/3125 | Loss: 0.4564
  Batch 800/3125 | Loss: 0.4255
  Batch 1000/3125 | Loss: 0.3973
  Batch 1200/3125 | Loss: 0.3084
  Batch 1400/3125 | Loss: 0.5300
  Batch 1600/3125 | Loss: 0.4606
  Batch 1800/3125 | Loss: 0.3499
  Batch 2000/3125 | Loss: 0.6910
 

In [10]:
torch.save(model.state_dict(), 'deepfake_detector.pth')
print("Model saved!")

# Download it to your PC
from google.colab import files
files.download('deepfake_detector.pth')

Model saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        outputs = model(images)
        predicted = (outputs >= 0.5).float()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Convert to flat lists
all_preds = [int(p[0]) for p in all_preds]
all_labels = [int(l[0]) for l in all_labels]

# Print results
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=['fake', 'real']))

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

Classification Report:
              precision    recall  f1-score   support

        fake       0.85      0.92      0.88     10000
        real       0.91      0.84      0.87     10000

    accuracy                           0.88     20000
   macro avg       0.88      0.88      0.88     20000
weighted avg       0.88      0.88      0.88     20000

Confusion Matrix:
[[9170  830]
 [1640 8360]]


In [13]:
import os

real_files = os.listdir('dataset/real_vs_fake/real-vs-fake/test/real')
fake_files = os.listdir('dataset/real_vs_fake/real-vs-fake/test/fake')

print("First 5 real images:", real_files[:5])
print("First 5 fake images:", fake_files[:5])

First 5 real images: ['52449.jpg', '03103.jpg', '31323.jpg', '11545.jpg', '04275.jpg']
First 5 fake images: ['ZWEJOHNX2Z.jpg', '9636K7E235.jpg', 'XESRKAKKAI.jpg', 'NVMFA7F52H.jpg', 'J5UDXZFC96.jpg']


In [14]:
predict_image('dataset/real_vs_fake/real-vs-fake/test/real/52449.jpg', model, device)
print("---")
predict_image('dataset/real_vs_fake/real-vs-fake/test/fake/ZWEJOHNX2Z.jpg', model, device)

Prediction : REAL
Confidence : 83.97%
---
Prediction : FAKE
Confidence : 83.99%


('FAKE', 0.8398783355951309)

Download the .pth and use it in folder.

In [ ]:
import torch

# Save in the old format that downloads cleanly
torch.save(model.state_dict(), 'deepfake_detector_v2.pth', _use_new_zipfile_serialization=False)

from google.colab import files
files.download('deepfake_detector_v2.pth')